# Low-rank query and KV compression

This notebook inspects the query and KV projection branches used by the educational content-only MLA layer.

In [1]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(7)

## Query compression

The query branch creates separate content and positional features.

In [2]:
from src.model import DeepSeekConfig
from src.model.mla import ContentOnlyMLA, KVCompression, QueryCompression

config = DeepSeekConfig()
x = torch.randn(2, 16, config.dim)

q_content, q_rope = QueryCompression(config)(x)
print("content query:", q_content.shape)
print("positional query:", q_rope.shape)

content query: torch.Size([2, 16, 8, 32])
positional query: torch.Size([2, 16, 8, 16])


## Joint KV compression

One latent is shared across heads, while learned up-projection slices reconstruct different content keys and values.

In [3]:
latent, k_rope, k_content, values = KVCompression(config)(x)
print("latent:", latent.shape)
print("positional key:", k_rope.shape)
print("content keys:", k_content.shape)
print("values:", values.shape)

latent: torch.Size([2, 16, 64])
positional key: torch.Size([2, 16, 16])
content keys: torch.Size([2, 16, 8, 32])
values: torch.Size([2, 16, 8, 32])


## Temporary content-only attention

The positional branches are returned but not used until decoupled RoPE is added.

In [4]:
mla = ContentOnlyMLA(config)
output = mla(x)
print("output:", output.shape)

output.square().mean().backward()
assert mla.q_branch.q_down.weight.grad is not None
assert mla.kv_branch.kv_down.weight.grad is not None
print("content-only MLA backward check passed")

output: torch.Size([2, 16, 256])
content-only MLA backward check passed
